# Fair Lending & Bias Detection

### Regulatory Compliance Tech Regtech — Banking-AI

AI models in banking must be fair. If an algorithm is more likely to deny loans to a specific group (e.g., based on gender or age), it's not just unethical—it's illegal under fair lending laws.

In this notebook:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of regulatory compliance and RegTech terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the regulatory compliance and RegTech problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** KYC automation, regulatory reporting, compliance monitoring, bias detection, and data privacy in banking.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Regulators require banks to prove their models don't discriminate. Fairness metrics are now as important as accuracy metrics.

**Input data used:** Credit score, income, debt-to-income ratio, and a protected attribute (Gender).

**What we predict:** Loan Approval (Yes/No).

**ML method used:** Logistic Regression.

**Learning goal:** Learn how to calculate fairness metrics and understand why "accuracy" isn't the only thing that matters.

**Primary stakeholders:** compliance officers, legal teams, audit managers, and data protection officers.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Dataset

We'll create a dataset where a bias is subtly (or not so subtly) injected.

In [ ]:
n = 2000
income = RNG.normal(50000, 15000, n)
credit_score = RNG.normal(650, 50, n)
gender = RNG.choice(['Male', 'Female'], n)

# Logic: Loan approved if credit_score > 670 and income > 40000
# BUT, let's inject a bias where 'Female' applicants have a slightly lower approval probability 
# for the same financial metrics (simulating historical bias in data).
approval_prob = (0.7 * (credit_score > 670) + 0.3 * (income > 45000))
approval_prob = np.clip(approval_prob, 0, 1)

# Apply bias
approval_prob = np.where(gender == 'Female', approval_prob * 0.85, approval_prob)

approved = (RNG.random(n) < approval_prob).astype(int)

df = pd.DataFrame({
    'income': income,
    'credit_score': credit_score,
    'gender': gender,
    'approved': approved
})

print(f"Dataset created with {n} applications.")
print(df.groupby('gender')['approved'].mean())

## Step 4 — Exploratory Data Analysis

EDA

Visualizing the approval rates by gender.

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='gender', y='approved', data=df, palette='magma')
plt.title('Loan Approval Rate by Gender')
plt.ylabel('Approval Probability')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "Loan Approval Rate by Gender". The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
df_encoded = pd.get_dummies(df, columns=['gender'], drop_first=True)
X = df_encoded.drop('approved', axis=1)
y = df_encoded['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2%}")

In [ ]:
test_df = X_test.copy()
test_df['actual'] = y_test
test_df['predicted'] = y_pred

approval_rate_male = test_df[test_df['gender_Male'] == 1]['predicted'].mean()
approval_rate_female = test_df[test_df['gender_Male'] == 0]['predicted'].mean()

disparate_impact = approval_rate_female / approval_rate_male

print(f"Approval Rate (Male): {approval_rate_male:.2%}")
print(f"Approval Rate (Female): {approval_rate_female:.2%}")
print(f"Disparate Impact Ratio: {disparate_impact:.3f}")

if disparate_impact < 0.8:
    print("CAUTION: Potential bias detected against female applicants.")
else:
    print("Model passes the 4/5ths fairness rule.")

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

To fix this, we might need to remove the 'gender' feature or use specialized fair-learning algorithms. Even if we remove 'gender', other features (like zip code or income) might act as proxies for it. RegTech tools help identify these hidden correlations.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end regulatory compliance and RegTech workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Automated compliance systems must be auditable and explainable to regulators.
- AI-driven surveillance can raise employee and customer privacy concerns.
- Bias in compliance models may lead to disproportionate scrutiny of certain groups.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in regulatory compliance and RegTech?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.